In [91]:
import numpy as np
import polars as pl
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import polars.selectors as cs


## Loading and Standardization

In [92]:
# Load the data
tep_fault_free = pl.read_parquet("../data/tep_fault_free_training.parquet")
tep_faulty = pl.read_parquet("../data/tep_faulty_training_runs01-20.parquet")

In [93]:
# Define global variables
RUN = "simulationRun"

In [94]:
# Define preprocessing functions
def channels(df): # Get the list of channels
    lst = df.drop(["faultNumber", "simulationRun", "sample"])
    lst_channels = lst.columns
    return lst_channels

In [95]:
def training_runs(df): # faulty free
    train = df.filter(pl.col(RUN) <= 300)
    return train

def validation_runs(df): # faulty free
    validation = df.filter((pl.col(RUN) >= 301) & (pl.col(RUN) <= 400))
    return validation

def test_runs(df): # fault free
    test = df.filter((pl.col(RUN) >= 401) & (pl.col(RUN) <= 500))
    return test

In [96]:
def standardization(df, lst_channels): # standardize the data set
    # compute the mean for each column
    train = training_runs(df)
    mean = []
    std_dev = []
    for channel in lst_channels:
        mean.append(train.select(pl.col(channel).mean()).item()) #get the exact value
        std_dev.append(train.select(pl.col(channel).std(ddof=1)).item()) 
    return mean, std_dev

def apply_standard(df, lst_channels, mean, std_dev): # take in any df (i.e. training, validation, test)
    standardized = df
    for i in range(len(lst_channels)):
        standardized = standardized.with_columns(pl.col(lst_channels[i]).sub(mean[i]) / std_dev[i])
    return standardized

In [97]:
def main():
    chan = channels(tep_fault_free) # list of channel names for tracked variables
    mean, std_dev = standardization(tep_fault_free, chan) # distribution of fault free data

    train = training_runs(tep_fault_free) # define training set (1-300)
    val = validation_runs(tep_fault_free) # define validation set (301-400)
    test = test_runs(tep_fault_free) # define test set (401-500)

    # Four standardized data frames for the training, validation, test, and faulty runs
    train_standardized = apply_standard(train, chan, mean, std_dev)
    val_standardized = apply_standard(val, chan, mean, std_dev)
    test_standardized = apply_standard(test, chan, mean, std_dev)
    faulty_standardized = apply_standard(tep_faulty, chan, mean, std_dev)
    


if __name__ == "__main__":
    main()

## PCA

In [98]:
# get data for PCA 
chan = channels(tep_fault_free) # list of channel names for tracked variables
mean, std_dev = standardization(tep_fault_free, chan) # distribution of fault free data

train = training_runs(tep_fault_free) # define training set (1-300)
val = validation_runs(tep_fault_free) # define validation set (301-400)
test = test_runs(tep_fault_free) # define test set (401-500)

# Four standardized data frames for the training, validation, test, and faulty runs
train_standardized = apply_standard(train, chan, mean, std_dev)
val_standardized = apply_standard(val, chan, mean, std_dev)
test_standardized = apply_standard(test, chan, mean, std_dev)
faulty_standardized = apply_standard(tep_faulty, chan, mean, std_dev)

In [99]:
metadata_cols = {"faultNumber", "simulationRun", "sample"}
feature_cols = [col for col in chan if col not in metadata_cols]

In [100]:
def PCA_model(df):
    "PCA model fitted on training data. Also visualizes covariance matrix"
    
    Z = df.select(feature_cols) # only want covariance under normal operation of the data, not the run numbers

    pca = PCA() # use PCA on the shortened test data
    pca.fit(Z.to_numpy()) # fit without column names, doesn't change anything on the training but avoids an error later
   
    """
    covariance = pca.get_covariance() # used to examine and see what variables are related 
    
    corr = pd.DataFrame(covariance, index=chan, columns=chan )
    fig, ax = plt.subplots(figsize=(14, 12))

    sns.heatmap(corr, cmap='coolwarm', xticklabels=True, yticklabels=True)
    ax.tick_params(labelsize=7)
    plt.title('Covariance Matrix Heatmap')
    plt.tight_layout()
    plt.show()
    """
    return(pca)

In [101]:
"""def score(pca, df):
    "For a given dataset (can be training, validation, testing) generate PCA score metrics"

    metadata = df.select(metadata_cols)
    Z = df.select(feature_cols)
    
    scaled_eigenvaues = pca.explained_variance_ratio_ # scaled, so they sum to one
    cumulative = np.cumsum(scaled_eigenvaues) # keeps a cumulative sum of eigenvalues
    k = int(np.argmax(cumulative >= 0.90) + 1) # counts up the number of scaled eigenvalues until it reaches 90%
    #print("k =", k)

    lam_k = pca.explained_variance_[:k] # raw eigenvalues of top k PCs
    P = pca.components_[:k].T # eigenvectors of the top k PCs
    
    #PPTz = pca.inverse_transform(pca.transform(Z)) incorrect because this should be fit along PCA 31 dimensions, not 52 sensors

    z = Z.to_numpy()  
      
    t_k = z @ P
    T2 = ((t_k ** 2) / lam_k).sum(axis=1)
    SPE = ((z - t_k @ P.T) ** 2).sum(axis=1)

    t_k_scores = pl.DataFrame(t_k, schema=[f"PC{i+1}" for i in range(k)])
    T2_score = pl.DataFrame(T2, schema=['T2'])
    SPE_score = pl.DataFrame(SPE, schema=['SPE'])
    Z = pl.concat([Z, t_k_scores, T2_score, SPE_score], how="horizontal_extend")

    return Z, metadata"""

'def score(pca, df):\n    "For a given dataset (can be training, validation, testing) generate PCA score metrics"\n\n    metadata = df.select(metadata_cols)\n    Z = df.select(feature_cols)\n\n    scaled_eigenvaues = pca.explained_variance_ratio_ # scaled, so they sum to one\n    cumulative = np.cumsum(scaled_eigenvaues) # keeps a cumulative sum of eigenvalues\n    k = int(np.argmax(cumulative >= 0.90) + 1) # counts up the number of scaled eigenvalues until it reaches 90%\n    #print("k =", k)\n\n    lam_k = pca.explained_variance_[:k] # raw eigenvalues of top k PCs\n    P = pca.components_[:k].T # eigenvectors of the top k PCs\n\n    #PPTz = pca.inverse_transform(pca.transform(Z)) incorrect because this should be fit along PCA 31 dimensions, not 52 sensors\n\n    z = Z.to_numpy()  \n\n    t_k = z @ P\n    T2 = ((t_k ** 2) / lam_k).sum(axis=1)\n    SPE = ((z - t_k @ P.T) ** 2).sum(axis=1)\n\n    t_k_scores = pl.DataFrame(t_k, schema=[f"PC{i+1}" for i in range(k)])\n    T2_score = pl.Da

In [102]:
def score(pca, df):
    """Calculate PCA T2 and SPE scores for each row in df."""
    metadata = df.select(metadata_cols)
    Z = df.select(feature_cols)
    z = Z.to_numpy()

    # Find the smallest number of components explaining at least 90% of variance.
    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
    k = int(np.argmax(cumulative_variance >= 0.90) + 1)

    # sklearn's transform centers each row using pca.mean_ before projecting it.
    t_k = pca.transform(z)[:, :k]
    eigenvalues = pca.explained_variance_[:k]

    T2 = ((t_k ** 2) / eigenvalues).sum(axis=1)

    # Reconstruct each row using only the first k components.
    reconstruction = t_k @ pca.components_[:k, :] + pca.mean_
    SPE = ((z - reconstruction) ** 2).sum(axis=1)

    t_k_scores = pl.DataFrame(t_k, schema=[f"PC{i+1}" for i in range(k)])
    T2_score = pl.DataFrame(T2, schema=['T2'])
    SPE_score = pl.DataFrame(SPE, schema=['SPE'])

    Z = pl.concat([Z, t_k_scores, T2_score, SPE_score], how="horizontal_extend")
    
    return Z, metadata

In [103]:
def write_scores_pca():
    chan = channels(tep_fault_free) # list of channel names for tracked variables
    mean, std_dev = standardization(tep_fault_free, chan)

    faulty_standardized = apply_standard(tep_faulty, chan, mean, std_dev) # 200k rows

    train = training_runs(tep_fault_free) # define training set (1-300)
    train_standardized = apply_standard(train, chan, mean, std_dev) #150k rows

    # train the PCA on the training data
    pca = PCA_model(train_standardized)

    # only score and export non-training data
    val = validation_runs(tep_fault_free) # define validation set (301-400)
    test = test_runs(tep_fault_free) # define test set (401-500)
    val_standardized = apply_standard(val, chan, mean, std_dev) # 50k rows
    test_standardized = apply_standard(test, chan, mean, std_dev) # 50k rows

    # for scoring, should be 300k rows (50k from val, 50k from test, 200k from faulty)
    fault_free_standardized = pl.concat([val_standardized, test_standardized], how="vertical")  # pl concat the two other dataframes

    fault_free_standardized_scored, fault_free_metadata = score(pca, fault_free_standardized) 
    faulty_standardized_scored, faulty_metadata = score(pca, faulty_standardized)

    # now add back metadata, remove PCA and data columns
    parquet_cols = ["faultNumber", "simulationRun", "sample", "T2", "SPE"]
    fault_free_pca_data_df = pl.concat([fault_free_metadata, fault_free_standardized_scored], how="horizontal_extend")
    faulty_pca_data_df = pl.concat([faulty_metadata, faulty_standardized_scored], how="horizontal_extend")

    fault_free_pca_df = fault_free_pca_data_df.select(parquet_cols)
    faulty_pca_df = faulty_pca_data_df.select(parquet_cols)

    scores_pca = pl.concat([fault_free_pca_df,faulty_pca_df], how="vertical")
    scores_pca.write_parquet(r"..\results\scores_pca.parquet")
    return scores_pca

In [104]:
scores_pca = write_scores_pca()


In [109]:
scores_pca

faultNumber,simulationRun,sample,T2,SPE
i64,i64,i64,f64,f64
0,301,1,3.43008,0.680933
0,301,2,12.749883,1.098481
0,301,3,17.112808,2.448329
0,301,4,14.603714,1.345962
0,301,5,12.629007,4.169891
…,…,…,…,…
20,20,496,39.979221,43.367803
20,20,497,42.353181,58.554436
20,20,498,51.243718,57.458983


In [110]:
tep_faulty.group_by("sample").len() #.sort("len", descending=True)

sample,len
i64,u32
265,400
399,400
149,400
140,400
15,400
…,…
500,400
244,400
387,400
